# DEMI | NOVA FCT

## Fatigue Design of Mechanical Joints and Aerospace Structures

##### José Xavier & Rui Martins

### Problems 11 — Review problems (mixed topics)

- [Exercise 11.1](#ex1) — Fracture toughness and fracture assessment
- [Exercise 11.2](#ex2) — Paris-law integration (review of P04)
- [Exercise 11.3](#ex3) — Miner's rule with three blocks
- [Exercise 11.4](#ex4) — Soderberg diagram for $R=0$
- [Exercise 11.5](#ex5) — ASTM E399 validity
- [Exercise 11.6](#ex6) — Residual strength diagram (point check)
- [Exercise 11.7](#ex7) — S–N estimation and Soderberg with $K_t$ (alloy selection)


In [10]:
import numpy as np
import matplotlib.pyplot as plt

FS = 14

# Reset to default style so any IDE dark-theme overrides do not bleed in.
plt.style.use('default')

plt.rcParams.update({
    'figure.figsize':   (10, 4.2), 'figure.dpi': 110,
    'font.family':      'serif',  'font.size': FS,
    'axes.grid':        True,     'grid.alpha': 0.30,
    'lines.linewidth':  2,
    # --- Backgrounds (white) ---
    'axes.facecolor':   'white',
    'figure.facecolor': 'white',
    'savefig.facecolor':'white',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black',
    # --- Force all text BLACK so the IDE dark theme does not hide it ---
    'text.color':       'black',
    'axes.labelcolor':  'black',
    'axes.edgecolor':   'black',
    'axes.titlecolor':  'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'xtick.labelcolor': 'black',
    'ytick.labelcolor': 'black',
    'legend.labelcolor':'black',
})
COLORS = dict(curve='#1f4e79', point='#c0392b', accent='#8e44ad',
              guide='0.55', warn='#e67e22', cool='#16a085')

SQRT1000 = np.sqrt(1000.0)

def Y_feddersen_DK(aW):
    aw = np.asarray(aW, dtype=float)
    return (1 - 0.1*aw**2 + 0.96*aw**4)*np.sqrt(1.0/np.cos(np.pi*aw))

def f_CT(aW):
    aw = np.asarray(aW, dtype=float)
    pref = (2.0 + aw)/(1.0 - aw)**1.5
    poly = 0.886 + 4.64*aw - 13.32*aw**2 + 14.72*aw**3 - 5.6*aw**4
    return pref*poly


<a id='ex1'></a>
---

## Exercise 11.1 — Fracture toughness and fracture assessment

An aircraft component is fabricated from an aluminium alloy with
$K_{\text{IC}} = 30.2\;\text{MPa}\sqrt{\text{m}}$. The design specifies that, in the
presence of an internal crack of total length $2a = 3$ mm, fracture shall only occur
for a tensile stress of $220$ MPa.

Determine:

(a) the geometry correction factor $Y$ for this design condition, rounded to one
decimal place;

(b) whether fracture will occur if a stress of $300$ MPa is applied in the presence
of an internal crack of total length $2a = 1$ mm.

### Solution

#### (a) Geometry correction $Y$

At fracture, $K_I = K_{IC}$:

\begin{equation*}
Y = \dfrac{K_{IC}}{\sigma\sqrt{\pi a}}
   = \dfrac{30.2}{220\sqrt{\pi\cdot 1.5\times 10^{-3}}}
   \approx 2.0.
\tag{11.1.1}\end{equation*}


In [11]:
K_IC = 30.2                         # MPa√m
sigma_f = 220.0
a = 1.5e-3                          # m (half-length of 2a = 3 mm)
Y = K_IC/(sigma_f*np.sqrt(np.pi*a))
print(f'(a) Y = K_IC / (σ_f · √(πa)) = {K_IC}/({sigma_f}·√(π·{a*1e3} mm))')
print(f'    Y ≈ {Y:.3f}  →  rounded to one decimal: Y = {round(Y,1)}')
Y = round(Y,1)


(a) Y = K_IC / (σ_f · √(πa)) = 30.2/(220.0·√(π·1.5 mm))
    Y ≈ 2.000  →  rounded to one decimal: Y = 2.0


#### (b) Fracture at $2a=1$ mm under $\sigma=300$ MPa?


In [12]:
a_new = 0.5e-3                     # m
sigma_new = 300.0
K_I = Y*sigma_new*np.sqrt(np.pi*a_new)
print(f'(b) K_I = Y·σ·√(πa) = {Y}·{sigma_new}·√(π·{a_new*1e3:.1f} mm)')
print(f'    K_I = {K_I:.2f} MPa√m  vs  K_IC = {K_IC} MPa√m')
print(f'    K_I/K_IC = {K_I/K_IC:.3f}  →  ', 'NO fracture' if K_I<K_IC else 'FRACTURE')


(b) K_I = Y·σ·√(πa) = 2.0·300.0·√(π·0.5 mm)
    K_I = 23.78 MPa√m  vs  K_IC = 30.2 MPa√m
    K_I/K_IC = 0.787  →   NO fracture


### Solution summary — Exercise 11.1

| Quantity | Value |
|---|---|
| Fracture toughness $K_{\mathrm{IC}}$ | 30.2 MPa√m |
| Crack half-length at design fracture ($2a=3$ mm) | $a = 1.5$ mm |
| Geometry correction factor $Y$ | **2.0** |
| $K_I$ at $\sigma=300$ MPa, $2a=1$ mm | 23.78 MPa√m |
| $K_I / K_{\mathrm{IC}}$ | 0.787 |
| Fracture at $\sigma=300$ MPa? | **No fracture** ($K_I < K_{\mathrm{IC}}$) |

<a id='ex2'></a>
---

## Exercise 11.2 — Paris-law integration

A CCT panel of width $W = 100$ mm is made from a carbon steel with
$K_{\text{IC}} = 50\;\text{MPa}\sqrt{\text{m}}$. The panel is subjected to
constant-amplitude loading with $\Delta\sigma = 200$ MPa and $R = 0.25$.
The initial crack has $2a_0 = 1$ mm. The Paris law is

$$
\frac{\mathrm{d}a}{\mathrm{d}N}
= 1\times10^{-15}\,(\Delta K)^3
\quad\text{(mm/cycle, N}\cdot\text{mm}^{-3/2}\text{)}.
$$


(a) Compute $\sigma_{\max}$;

(b) Describe the numerical procedure for computing $N_f$. What are the key steps?

(c) Verify the order of magnitude of $N_f$ by computing it for an infinite plate ($Y = 1$) using the closed-form Paris integral. Compare with the finite-width result $N_f \approx 5.26\times10^7$ cycles.

### Solution

#### (a) $\sigma_{\max}$

\begin{equation*}
\sigma_{\max} = \dfrac{\Delta\sigma}{1-R} = \dfrac{200}{0.75} \approx 266.7~\text{MPa}.
\tag{11.2.1}\end{equation*}

#### (b) Numerical procedure

1. Compute $a_c$ from $Y(a_c/W)\,\sigma_{\max}\sqrt{\pi a_c} = K_{IC}$ (bisection).
2. From $a_0$ to $a_c$ step by $\Delta a$:
   * compute $Y(a/W)$ and $\Delta K = Y\,\Delta\sigma\sqrt{\pi a}$,
   * Paris rate $\mathrm{d}a/\mathrm{d}N = C(\Delta K)^{m}$,
   * advance: $\Delta N = \Delta a/(\mathrm{d}a/\mathrm{d}N)$, $a\!\leftarrow\!a+\Delta a$, $N\!\leftarrow\!N+\Delta N$.
3. $N_f = N$ at the end.

#### (c) Closed-form check for the infinite plate ($Y=1$)

\begin{equation*}
N_f^{(Y=1)} = \dfrac{1}{C\,(\Delta\sigma)^{m}\,\pi^{m/2}}\,
              \dfrac{a_0^{1-m/2}-a_c^{1-m/2}}{m/2-1}.
\tag{11.2.2}\end{equation*}

We use the *same* $a_c$ as in the finite-width problem (the closed form
isolates the geometry effect from the limits).


In [13]:
W, R = 100.0, 0.25
dsigma = 200.0
sigma_max = dsigma/(1-R)
print(f'(a) σ_max = Δσ/(1-R) = {sigma_max:.2f} MPa')

K_IC = 50.0; K_IC_Nmm = K_IC*SQRT1000
C, m = 1e-15, 3.0
a0 = 0.5  # mm

def K_max_Nmm(a_mm):
    return float(Y_feddersen_DK(a_mm/W))*sigma_max*np.sqrt(np.pi*a_mm)

lo, hi = a0, 0.499*W
for _ in range(80):
    mid = 0.5*(lo+hi)
    if K_max_Nmm(mid) < K_IC_Nmm: lo = mid
    else:                          hi = mid
a_c = 0.5*(lo+hi)
print(f'    a_c = {a_c:.2f} mm  (2a_c = {2*a_c:.2f} mm)')

# Closed-form for Y=1 (in mm, dK in N·mm^-1.5)
# da/dN = C ( Δσ √(π a) )^m  with σ in MPa·(mm)^(1/2) gives dK in N·mm^-1.5
# So no extra conversion needed.
factor = 1.0/(C*(dsigma**m)*(np.pi**(m/2)))
Nf_Y1 = factor*(a0**(1-m/2) - a_c**(1-m/2))/(m/2 - 1)
print(f'(c) Closed-form for Y=1: N_f = {Nf_Y1:.3e} cycles')

# Numerical with Feddersen Y
def integrate_CCT(a0, ac, da):
    a, N = a0, 0.0
    while a < ac:
        Y_ = float(Y_feddersen_DK(a/W))
        dK = Y_*dsigma*np.sqrt(np.pi*a)
        dadN = C*dK**m
        step = min(da, ac-a)
        N += step/dadN
        a += step
    return N

Nf_fed = integrate_CCT(a0, a_c, 0.1)
print(f'    Numerical with Feddersen Y: N_f ≈ {Nf_fed:.3e} cycles')
print(f'    Ratio  N_f(Y=1) / N_f(Feddersen) = {Nf_Y1/Nf_fed:.3f}   '
      '(Y(a/W) > 1 throughout → Y=1 over-estimates life)')


(a) σ_max = Δσ/(1-R) = 266.67 MPa
    a_c = 10.60 mm  (2a_c = 21.19 mm)
(c) Closed-form for Y=1: N_f = 4.970e+07 cycles
    Numerical with Feddersen Y: N_f ≈ 5.264e+07 cycles
    Ratio  N_f(Y=1) / N_f(Feddersen) = 0.944   (Y(a/W) > 1 throughout → Y=1 over-estimates life)


### Solution summary — Exercise 11.2

| Quantity | Value |
|---|---|
| $\sigma_{\max} = \Delta\sigma/(1-R)$ | 266.67 MPa |
| Critical crack half-length $a_c$ (Feddersen, $K_{\max}=K_{\mathrm{IC}}$) | 10.60 mm ($2a_c = 21.19$ mm) |
| $N_f$ — closed form ($Y=1$, infinite plate) | 4.970×10⁷ cycles |
| $N_f$ — numerical with Feddersen $Y(a/W)$ | **5.264×10⁷ cycles** |
| Ratio $N_f(Y{=}1) / N_f(\text{Feddersen})$ | 0.944 (Feddersen $Y>1$ → shorter life) |

<a id='ex3'></a>
---

## Exercise 11.3 — Miner's rule with three blocks

An AISI 1045 steel component has $\sigma_f' = 1198.6$ MPa and $b = -0.09$.
It is subjected to the following loading spectrum:

| Block | $\sigma_a$ (MPa) | $\sigma_m$ (MPa) | $n$ (cycles) |
|---:|---:|---:|---:|
| 1 | 300 | 150 | 300 000 |
| 2 | 300 | 0 | 150 000 |
| 3 | 350 | 150 | 30 000 |

Using the modified Basquin relation

$$
\sigma_a = (\sigma_f' - \sigma_m)(2N_f)^b,
$$

compute:

(a) $N_{f,i}$ for each block;

(b) the total Miner damage $D$;

(c) whether the component survives.

### Solution

In [14]:
sigma_fp, b = 1198.6, -0.09
blocks = [(300, 150, 300_000),
          (300,   0, 150_000),
          (350, 150,  30_000)]
print(f'{"Block":>5} {"σ_a":>5} {"σ_m":>5} {"N_f":>14} {"n/N_f":>12}')
D = 0.0
for i,(sa, sm, n) in enumerate(blocks,1):
    Nf = 0.5*(sa/(sigma_fp - sm))**(1.0/b)
    d  = n/Nf
    D += d
    print(f'{i:>5d} {sa:>5.0f} {sm:>5.0f} {Nf:>14.3e} {d:>12.4f}')
print(f'\n(b) Total Miner damage  D = {D:.4f}')
print('(c)', 'survives — D < 1' if D < 1 else 'fails during the spectrum')
if D < 1:
    print(f'    Number of additional identical spectra to reach D=1: ≈ {1/D - 1:.2f}')


Block   σ_a   σ_m            N_f        n/N_f
    1   300   150      5.467e+05       0.5488
    2   300     0      2.415e+06       0.0621
    3   350   150      9.860e+04       0.3043

(b) Total Miner damage  D = 0.9151
(c) survives — D < 1
    Number of additional identical spectra to reach D=1: ≈ 0.09


### Solution summary — Exercise 11.3

| Block | $\sigma_a$ (MPa) | $\sigma_m$ (MPa) | $N_{f,i}$ | $n_i/N_{f,i}$ |
|---:|---:|---:|---:|---:|
| 1 | 300 | 150 | 5.467×10⁵ | 0.5488 |
| 2 | 300 | 0 | 2.415×10⁶ | 0.0621 |
| 3 | 350 | 150 | 9.860×10⁴ | 0.3043 |
| — | — | **Total damage $D$** | — | **0.9151** |

**Verdict: component survives** ($D = 0.9151 < 1$). Approximately 0.09 additional identical spectra remain before failure.

<a id='ex4'></a>
---

## Exercise 11.4 — Soderberg diagram for $R=0$

A forged steel has $\sigma_e = 386$ MPa and $\sigma_{f0} = 296$ MPa
for fully reversed loading ($R=-1$).

Determine:

(a) the fatigue limit, expressed as stress amplitude, for pulsating loading
($R=0$), using the Soderberg relation;

(b) the corresponding value of $\sigma_{\max}$.

### Solution

For $R=0$, $\sigma_m=\sigma_a$. Substitute in Soderberg:

\begin{equation*}
\sigma_a = \dfrac{1}{1/\sigma_{f0}+1/\sigma_e}.
\tag{11.4.1}\end{equation*}


In [15]:
sigma_e, sigma_f0 = 386.0, 296.0
sa_R0 = 1.0/(1/sigma_f0 + 1/sigma_e)
sm_R0 = sa_R0
smax_R0 = sa_R0 + sm_R0
print(f'(a) σ_a^max (R=0) = {sa_R0:.2f} MPa')
print(f'(b) σ_max = σ_a + σ_m = 2 σ_a = {smax_R0:.2f} MPa')


(a) σ_a^max (R=0) = 167.53 MPa
(b) σ_max = σ_a + σ_m = 2 σ_a = 335.06 MPa


### Solution summary — Exercise 11.4

| Quantity | Value |
|---|---|
| Fatigue limit ($R=-1$), $\sigma_{f0}$ | 296.0 MPa |
| Ultimate stress $\sigma_e$ | 386.0 MPa |
| Max. stress amplitude ($R=0$, Soderberg), $\sigma_a^{\max}$ | **167.53 MPa** |
| Max. peak stress ($R=0$), $\sigma_{\max} = 2\,\sigma_a^{\max}$ | **335.06 MPa** |

<a id='ex5'></a>
---

## Exercise 11.5 — ASTM E399 validity

A CT specimen with $B=50$ mm, $W=100$ mm and $a=52$ mm, made of a forged steel
with $\sigma_e = 1050$ MPa, yields
$K_Q = 156.8\;\text{MPa}\sqrt{\text{m}}$.

Determine:

(a) whether the ASTM E399 validity conditions are satisfied for both the
thickness $B$ and the remaining ligament $(W-a)$, where each must satisfy

$$
B,\; W-a \geq 2.5\left(\frac{K_Q}{\sigma_e}\right)^2
$$

(b) Is $K_Q$ a valid $K_{\text{IC}}$? If not, what is the maximum $K_Q$ that would be valid with this specimen?

### Solution


In [16]:
B, W, a, sigma_e, K_Q = 50.0, 100.0, 52.0, 1050.0, 156.8
ligament = W - a
min_req_mm = 2.5*(K_Q/sigma_e)**2 * 1e3
print(f'2.5·(K_Q/σ_e)² = {min_req_mm:.2f} mm')
print(f'  Thickness B  = {B} mm   {"≥" if B>=min_req_mm else "<"} requirement')
print(f'  Ligament W-a = {ligament} mm  {"≥" if ligament>=min_req_mm else "<"} requirement')
valid = (B >= min_req_mm) and (ligament >= min_req_mm)
print('Verdict:', 'VALID K_IC' if valid else 'NOT valid')

K_Q_max = sigma_e*np.sqrt(min(B, ligament)*1e-3 / 2.5)
print(f'Max K_Q valid with present specimen: {K_Q_max:.2f} MPa√m  (actual K_Q = {K_Q} MPa√m)')


2.5·(K_Q/σ_e)² = 55.75 mm
  Thickness B  = 50.0 mm   < requirement
  Ligament W-a = 48.0 mm  < requirement
Verdict: NOT valid
Max K_Q valid with present specimen: 145.49 MPa√m  (actual K_Q = 156.8 MPa√m)


### Solution summary — Exercise 11.5

| Dimension | Actual | Requirement $2.5(K_Q/\sigma_e)^2$ | Valid? |
|---|---:|---:|---:|
| Thickness $B$ | 50.0 mm | 55.75 mm | **No** |
| Ligament $W-a$ | 48.0 mm | 55.75 mm | **No** |

**$K_Q = 156.8$ MPa√m is NOT a valid $K_{\mathrm{IC}}$.**
Maximum valid $K_Q$ for this specimen: **145.49 MPa√m**.

<a id='ex6'></a>
---

## Exercise 11.6 — Residual strength diagram

For a CCT panel with $W = 200$ mm, made of an aluminium alloy with
$K_{\text{IC}} = 37.2\;\text{MPa}\sqrt{\text{m}}$ and $\sigma_e = 450$ MPa,
consider a crack of total length $2a = 20$ mm.

Determine:

(a) the LEFM fracture stress using the Feddersen correction;

(b) the net-section plastic collapse stress;

(c) which failure mode governs for this crack size.

### Solution


In [17]:
W, K_IC, sigma_e = 200.0, 37.2, 450.0
a = 10.0   # mm (half-length)
aW = a/W
Y = float(Y_feddersen_DK(aW))
sigma_LEFM = K_IC/(Y*np.sqrt(np.pi*a*1e-3))     # MPa
sigma_PL   = sigma_e*(1 - 2*a/W)
print(f'(a) LEFM  : Y = {Y:.4f},  σ_f^LEFM = K_IC/(Y·√(πa)) = {sigma_LEFM:.2f} MPa')
print(f'(b) Net-section yield  σ_f^PL = σ_e (1-2a/W) = {sigma_PL:.2f} MPa')
print(f'(c) Governing:', 'LEFM (brittle)' if sigma_LEFM < sigma_PL else 'plastic collapse')


(a) LEFM  : Y = 1.0060,  σ_f^LEFM = K_IC/(Y·√(πa)) = 208.63 MPa
(b) Net-section yield  σ_f^PL = σ_e (1-2a/W) = 405.00 MPa
(c) Governing: LEFM (brittle)


### Solution summary — Exercise 11.6

| Quantity | Value |
|---|---|
| Crack half-length $a$ | 10.0 mm |
| Feddersen correction $Y(a/W)$ | 1.0060 |
| LEFM fracture stress $\sigma_f^{\mathrm{LEFM}}$ | **208.63 MPa** |
| Net-section plastic collapse $\sigma_f^{\mathrm{PL}} = \sigma_e(1-2a/W)$ | 405.00 MPa |
| Governing failure mode | **LEFM (brittle fracture)** |

<a id='ex7'></a>
---

## Exercise 11.7 — S–N estimation and Soderberg with $K_t$ (alloy selection)

Three aluminium alloys are candidates for a shaft of diameter $d$ subjected to
rotating bending with a superimposed static axial load. The shaft has a shoulder
with $K_t = 2.5$ and $q = 0.85$. The loading gives $\sigma_a = 120$ MPa and
$\sigma_m = 80$ MPa.

| Alloy | $\sigma_e$ (MPa) | $\sigma_u$ (MPa) |
|---|---:|---:|
| Al 2024-T3 | 345 | 485 |
| Al 6061-T6 | 275 | 310 |
| Al 7075-T6 | 505 | 572 |

Determine:

(a) for each alloy, $\sigma_{f0}$ estimated from $\sigma_u$, using
$\sigma_{f0} \approx 0.35\,\sigma_u$ for aluminium;

(b) $K_f = 1 + q(K_t - 1)$;

(c) which alloys provide a safe design according to the Soderberg criterion,
using $\sigma_{f0}/K_f$ and $\sigma_e$.

### Solution


In [18]:
sa_app, sm_app = 120.0, 80.0
K_t, q = 2.5, 0.85
K_f = 1 + q*(K_t - 1)
print(f'(b) K_f = 1 + q(K_t-1) = 1 + {q}·{K_t-1} = {K_f:.3f}')

alloys = [
    ('Al 2024-T3', 345.0, 485.0),
    ('Al 6061-T6', 275.0, 310.0),
    ('Al 7075-T6', 505.0, 572.0),
]
print(f'\n{"Alloy":<12} {"σ_u":>5} {"σ_e":>5} {"σ_f0=0.35σ_u":>14} '
      f'{"σ_f0/K_f":>11} {"LHS (Soderberg)":>17} {"safe?":>7}')
for name, se, su in alloys:
    sf0 = 0.35*su
    sf0_eff = sf0/K_f
    LHS = sa_app/sf0_eff + sm_app/se
    safe = LHS <= 1.0
    print(f'{name:<12} {su:>5.0f} {se:>5.0f} {sf0:>14.2f} {sf0_eff:>11.2f} '
          f'{LHS:>17.3f} {("YES" if safe else "NO"):>7}')


(b) K_f = 1 + q(K_t-1) = 1 + 0.85·1.5 = 2.275

Alloy          σ_u   σ_e   σ_f0=0.35σ_u    σ_f0/K_f   LHS (Soderberg)   safe?
Al 2024-T3     485   345         169.75       74.62             1.840      NO
Al 6061-T6     310   275         108.50       47.69             2.807      NO
Al 7075-T6     572   505         200.20       88.00             1.522      NO


**Discussion.** With $K_f = 2.275$, the effective fatigue limits $\sigma_{f0}/K_f$ drop to 74.6, 47.7, and 88.0 MPa for Al 2024-T3, Al 6061-T6, and Al 7075-T6 respectively — all well below the applied $\sigma_a = 120$ MPa. As a result, **none of the three alloys satisfies the Soderberg criterion** (LHS = 1.84, 2.81, 1.52). Al 7075-T6 gives the lowest Soderberg ratio (closest to 1) because it combines the highest $\sigma_{f0}$ and $\sigma_e$, but it still does not pass. To achieve a safe design one would need to: (i) reduce the notch severity (larger fillet → lower $K_f$), (ii) reduce the applied stress amplitudes by increasing the shaft cross-section, or (iii) select a higher-strength alloy with $\sigma_u \gtrsim 900$ MPa.

The notch-correction $K_f$ is the *single most influential* parameter: a 10 % reduction in $K_f$ (e.g. via a generous fillet radius) lowers the Soderberg LHS by roughly 9 % — bringing Al 7075-T6 to LHS ≈ 1.38, still not safe, but clearly pointing toward the required design change.

### Solution summary — Exercise 11.7

$K_f = 1 + q(K_t-1) = 2.275$; loading: $\sigma_a = 120$ MPa, $\sigma_m = 80$ MPa.

| Alloy | $\sigma_u$ (MPa) | $\sigma_e$ (MPa) | $\sigma_{f0}=0.35\sigma_u$ | $\sigma_{f0}/K_f$ | LHS (Soderberg) | Safe? |
|---|---:|---:|---:|---:|---:|---:|
| Al 2024-T3 | 485 | 345 | 169.75 MPa | 74.62 MPa | 1.840 | **No** |
| Al 6061-T6 | 310 | 275 | 108.50 MPa | 47.69 MPa | 2.807 | **No** |
| Al 7075-T6 | 572 | 505 | 200.20 MPa | 88.00 MPa | 1.522 | **No** |

**None of the three alloys satisfies the Soderberg criterion** under the given loading with $K_f = 2.275$.

---

### Final remarks

* This problem set is a **review**: it does not introduce new theory but
  exercises the four pillars of fatigue assessment (toughness, Paris
  integration, Miner's rule, Soderberg) in a single sitting.
* The numerical answers for Ex 11.1 (Y=2.0), Ex 11.2 (Y=1 closed-form vs
  finite-width), Ex 11.4 ($\sigma_a^{\max}(R=0)\approx 167$ MPa) and Ex 11.5
  (K_Q invalid) should match the canonical textbook solutions to within
  rounding.


---

Copyright (c) DEMI - NOVA FCT

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>

Parts of this notebook were prepared with AI assistance (<a href="https://www.anthropic.com/claude-code" target="_blank">Claude Code</a>, Anthropic), reviewed and verified by the author.